In [1]:
import torch, math, tiktoken, time, inspect
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from datetime import datetime

# from transformers.utils import logging
# logging.set_verbosity_error()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

current_time = datetime.now().strftime("%d%m%Y-%H%M%S")
writer = SummaryWriter(f'runs/GPT2_{current_time}')


NVIDIA GeForce GTX 1650


In [2]:
class GPTConfig:
    batch_size: int = 2
    block_size: int = 1024
    vocab_size: int = 50257
    # vocab_size: int = 50304 # changed to nice number
    n_embd: int = 768
    n_head: int = 12
    n_layer: int = 12
config = GPTConfig()


In [3]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, config.n_embd * 4)
        self.gelu = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(config.n_embd * 4, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))


In [4]:
class CasualSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        # self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
        # .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimension
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        
        # att = (q @ k.transpose(-2, -1)) * (1. / math.sqrt(k.size(-1)))
        # att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float("-inf"))
        # att = F.softmax(att, dim=-1)
        # y = att @ v # (B, nh, T, T) @ (B, nh, T, hs) -> (B, nh, T, hs)

        y = F.scaled_dot_product_attention(q,k,v, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side
        return self.c_proj(y) # (B, T, C)
        

In [5]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CasualSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)
    
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


In [6]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(
            {
                "wte": nn.Embedding(config.vocab_size, config.n_embd),
                "wpe": nn.Embedding(config.block_size, config.n_embd),
                "h": nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
                "ln_f": nn.LayerNorm(config.n_embd),
            }
        )
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        std = 0.02
        if hasattr(module, "NANOGPT_SCALE_INIT"):
            std *= (2 * self.config.n_layer) ** -0.5
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is  not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module,  nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)


    def forward(self, idx,  y=None):
        B, T = idx.size()
        assert T <= self.config.block_size, f"cannot forward sequence of length {T}, block size"

        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        pos_emb = self.transformer.wpe(pos)
        tok_emb = self.transformer.wte(idx)
        x = tok_emb + pos_emb

        for block in self.transformer.h:
            x = block(x)
        
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if y is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
        return logits, loss

    @classmethod
    def from_pretrained(cls, model_type):
        assert model_type in {"gpt2", "gpt2-medium", "gpt2-large", "gpt2-xl"}
        from transformers.models.gpt2 import GPT2LMHeadModel
        print(f"loading weights from pretrained gpt: {model_type}")

        config_args = {
            'gpt2': dict(n_layer=12, n_head=12, n_embd=768),          # 124M   Parameter
            'gpt2-medium': dict(n_layer=24, n_head=16, n_embd=1024),  # 350M   Parameter
            'gpt2-large': dict(n_layer=36, n_head=20, n_embd=1280),   # 774M   Parameter
            'gpt2-xl': dict(n_layer=48, n_head=25, n_embd=1600),      # 1.558B Parameter
        }[model_type]

        config_args['vocab_size'] = 50257
        config_args['block_size'] = 1024
        config = GPTConfig()
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard the mask

        sd_hf = GPT2LMHeadModel.from_pretrained(model_type).state_dict()

        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # discard the mask
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # discard the mask
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']

        assert len(sd_keys_hf) == len(sd_keys), f"Mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])
        return model

    def configure_optimizers(self, weight_decay, learning_rate, device):
        param_dict = {pn: p for pn, p in self.named_parameters()}
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}

        decay_params = [p for n, p in param_dict.items() if p.dim()>=2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim()<2]

        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0},
        ]
        num_decay_params = sum(p.numel() for p in decay_params)
        num_nodecay_params = sum(p.numel() for p in nodecay_params)
        print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params:,} parameters")
        print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params:,} parameters")
        # Create AdamW optimizer and use the fused version if it is available
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device == 'cuda'
        extra_args = dict(fused=True) if use_fused else dict()
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.95), eps=1e-8, **extra_args)
        print(f"using fused AdamW: {use_fused}")

        return optimizer


In [7]:
class Mydataset(Dataset):
    def __init__(self, data, context_length):
        self.tokens = torch.tensor(data)
        self.context_length = context_length
        
        print(f"loaded {len(self.tokens)} tokens")

    def __len__(self):
        return len(self.tokens) - self.context_length

    def __getitem__(self, idx):
        x = self.tokens[idx : idx + self.context_length]
        y = self.tokens[idx + 1 : idx + self.context_length + 1]
        return x, y

with open('Data/harrypotter.txt', 'r') as f:
    text = f.read()
enc = tiktoken.get_encoding('gpt2')
tokens = enc.encode(text)

div = int(len(tokens) * 0.95)
train_loader = DataLoader(Mydataset(tokens[:div], config.block_size), config.batch_size, shuffle=True, drop_last=True)
test_loader  = DataLoader(Mydataset(tokens[div:], config.block_size), config.batch_size, shuffle=False, drop_last=True)

torch.set_float32_matmul_precision('high') # optimization


loaded 1695456 tokens
loaded 89235 tokens


In [8]:
model = GPT.from_pretrained('gpt2')
# model = GPT(config)
model.to(device)
# model = torch.compile(model)
print(f"total parameter: {sum(p.numel() for p in model.parameters())}")


loading weights from pretrained gpt: gpt2


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


total parameter: 124439808


In [9]:
# optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
optimizer = model.configure_optimizers(weight_decay=0.1, learning_rate=6e-4, device=device)


num decayed parameter tensors: 50, with 124,318,464 parameters
num non-decayed parameter tensors: 98, with 121,344 parameters
using fused AdamW: True


In [10]:
max_lr = 3e-4
min_lr = max_lr * 0.1
warmup_steps = 10
max_steps = 50
def get_lr(it):
    if it < warmup_steps:
        return max_lr * (it + 1) / warmup_steps
    if it > max_steps:
        return min_lr
    decay_ratio = (it - warmup_steps) / (max_steps - warmup_steps)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)


In [11]:
def save_model(model, filepath):
    torch.save(model.state_dict(), filepath)
    print(f"Model saved to {filepath}")
    torch.cuda.empty_cache()
def generate_text(
    model,
    enc,
    device,
    start_text="",
    max_new_tokens=200,
    temperature=1.0,
    top_k=None,
):
    model.eval()

    # encode prompt
    tokens = enc.encode(start_text)
    x = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        for _ in range(max_new_tokens):

            # crop context if too long
            x_cond = x[:, -model.config.block_size:]

            logits, _ = model(x_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("Inf")

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            x = torch.cat((x, next_token), dim=1)

    generated = enc.decode(x[0].tolist())
    return generated
    
def log_generation(model, enc, writer, epoch, device):
    sample = generate_text(
        model,
        enc,
        device,
        start_text="\n",
        max_new_tokens=300,
        temperature=0.9,
        top_k=50,
    )
    writer.add_text("Sample Generation", sample, epoch)
def evaluate(model, test_loader, device):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)

            logits, loss = model(x, y)

            total_loss += loss.item() * x.size(0)
            total_tokens += x.size(0)

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)

    return avg_loss, perplexity


In [12]:
save_model(model, f"Models/GPT2-Pretrained.pth")


Model saved to Models/GPT2-Pretrained.pth


In [15]:
generate_text(model, enc, writer, 0, device)
for step in range(max_steps):
    model.train()
    # seat custom learning rate    
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr 

    # training loop
    tqdm_bar = tqdm(train_loader)
    for i, (x, y) in enumerate(tqdm_bar):
        x, y = x.to(device) , y.to(device)
        
        with torch.autocast(device_type=device, dtype=torch.float16):
            logits, loss = model(x, y)

        optimizer.zero_grad()
        loss.backward()
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    
        optimizer.step()

        idx = step * len(tqdm_bar) + i
        writer.add_scalar('Training Loss', loss.item(), idx)
        writer.add_scalar('Learning Rate', optimizer.param_groups[0]['lr'], idx)
        writer.add_scalar('Norm', norm, idx)

        tqdm_bar.set_description(f"Epoch: {step+1}")
    
    # ---- evaluation ----
    avg_test_loss, ppl = evaluate(model, test_loader, device)

    writer.add_scalar("Test Loss", avg_test_loss, step)
    writer.add_scalar("Perplexity", ppl, epoch)

    log_generation(model, enc, writer, step+1, device)
    save_model(model, f"Models/GPT2-{step+1}.pth")


Epoch: 1:   0%|          | 20/847216 [01:59<1403:43:56,  5.96s/it]


KeyboardInterrupt: 

In [16]:
def continue_generation(
    model,
    enc,
    device,
    prompt: str,
    max_new_tokens: int = 200,
    temperature: float = 1.0,
    top_k: int | None = 50,
):
    was_training = model.training
    model.eval()

    # Encode prompt
    tokens = enc.encode(prompt)
    x = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        for _ in range(max_new_tokens):

            # Crop to context window
            x_cond = x[:, -model.config.block_size:]

            logits, _ = model(x_cond)
            logits = logits[:, -1, :] / temperature

            # Optional top-k filtering
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("Inf")

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            x = torch.cat((x, next_token), dim=1)

    # Decode full sequence
    output_text = enc.decode(x[0].tolist())

    if was_training:
        model.train()

    return output_text


In [20]:
while True:
    user_input = input("\nPrompt: ")
    if user_input.lower() == "exit":
        break

    result = continue_generation(
        model,
        enc,
        device,
        prompt=user_input,
        max_new_tokens=200,
        temperature=0.9,
        top_k=50
    )

    print("\nGenerated:\n")
    print(result)



Generated:

hello.com | @JimLWiglesworth<|endoftext|>In an interview with WGRI's "Duck Dynasty," ex-Pelosi Republican operative Karl Rove slammed President Trump for having "nothing to do" with the ongoing investigations into his financial dealings. (Published Monday, Aug. 8, 2017)

Former vice presidential candidate Karl Rove said he thinks President Donald Trump should stay away from the ongoing investigations into his controversial financial dealings.

"He should have nothing to do," Rove said on "Duck Dynasty."

Republican presidential challenger Donald Trump denied making false statements on a financial disclosure form for his business consulting business. (Published Tuesday, Aug. 7, 2017)

Rove said he believes Trump should stay away from the ongoing investigations into his financial affairs."He should have nothing to do."

"I do not believe he should be involved in any ongoing political activities," he said Tuesday on "Duck Dynasty." (Published Tuesday, Aug. 7,

Generated:

wha

IndexError: index -1 is out of bounds for dimension 1 with size 0